<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/appendix_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch入門

## A.2テンソルを理解する
- テンソルはその次数（階数）で特徴づけることができる数学的なオブジェクト
- スカラー（単なる数値）は階数0のテンソル
- ベクトルは階数1のテンソル
- 行列は階数2のテンソル

In [ ]:
# PyTorchテンソルを作成する
import torch

tensor0d = torch.tensor(1) # 0次元テンソル（スカラー）を作成

tensor1d = torch.tensor([1, 2, 3]) # 1次元テンソル（ベクトル）を作成

tensor2d = torch.tensor([[1, 2],
                         [3, 4]]) # 2次元テンソルを作成

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]]) # 3次元テンソルを作成

In [ ]:
# テンソルのデータ型
print(tensor1d.dtype)

In [ ]:
# 浮動小数点からテンソルを作成
# 32ビットの精度のテンソルを作成

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

- 32ビット浮動小数点は、64ビット浮動小数点よりも少ないメモリリソースと計算リソースでほとんどのディープラーニングに十分な精度を実現
- GPUアーキテクチャは32ビット計算に合わせて最適化されているため、このデータ型を使用するとモデルの訓練と推論を大幅に高速化できる
- テンソルの`to()` メソッドを使って精度の変更もできる

In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

In [ ]:
# PyTorchの一般的なテンソル演算
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])
print(tensor2d)

In [ ]:
print(tensor2d.shape)

In [ ]:
# reshape
print(tensor2d.reshape(3, 2))

In [ ]:
# reshapeより view()メソッドの方が一般的
print(tensor2d.view(3, 2))

- `view()` の場合は元のデータが連動していなければならず、そうでない場合エラーになる
- `reshape()` の場合は元の形状に関係なく動作し、必要であればデータをコピーすることで、目的の形状を確保する

In [ ]:
# テンソルの転置
print(tensor2d)
print(tensor2d.T)

In [ ]:
# 行列の乗算（mamul()）
print(tensor2d.matmul(tensor2d.T))

In [ ]:
# @演算子での行列乗算
print(tensor2d @ tensor2d.T)

## A.3 モデルを計算グラフとしてとらえる
- 自動微分エンジンであるautograd
- 動的計算グラフを使用して勾配を自動で計算する
- 計算グラフ：数式の表現と可視化を可能にする有効グラフのこと
- 誤差逆伝播法に必要な勾配を計算するのに使用する

In [ ]:
# ロジスティック回帰のフォワードパス
import torch.nn.functional as F

y = torch.tensor([1.0]) # 正解ラベル
x1 = torch.tensor([1.1]) # 入力特徴量
w1 = torch.tensor([2.2]) # 重みパラメータ
b = torch.tensor([0.0]) # バイアスユニット
z = x1 * w1 + b # 総入力
a = torch.sigmoid(z) # 活性化と出力
loss = F.binary_cross_entropy(a, y)

## A.4 自動微分は行内の計算を簡易化する
- 終端（葉）ノードでrequired_grad属性がTrueに設定されていれば、デフォルトで内部に計算グラフが構築される
- 連鎖律で誤差逆伝播を考える
- 偏微分：ある関数がその変数の1つに対してどのように変化するかを表す尺度
- 勾配：多変量関数の偏微分をすべて含んでいるベクトル
- 多変量関数：入力として複数の変数を持つ関数

In [ ]:
# autogradによる勾配の計算
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

# 計算グラフを残すためにretain_graf=Trueを指定
# デフォルトはメモリ解放するために計算グラフを削除する
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [ ]:
# モデルパラメータ
print(grad_L_w1)
print(grad_L_b)

In [ ]:
# 自動化
loss.backward()
print(w1.grad)
print(b.grad)

## A.5 多層ニューラルネットワークを実装する
- 多層パーセプトロン：全結合ネットワーク

In [ ]:
# 2つの隠れ層をもつ多層パーセプトロン
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1つ目の隠れ層
            torch.nn.Linear(num_inputs, 30),
            # 非線形活性化関数は隠れ層の間に置かれる
            torch.nn.ReLU(),

            # 2つ目の隠れ層
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # 出力層
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)

        # 最後の層の出力はロジットと呼ばれる
        return logits

In [ ]:
# インスタンス化
model = NeuralNetwork(50, 3)

In [ ]:
print(model)

In [ ]:
# パラメータの総数を確認
num_paramas = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {num_paramas}")

In [ ]:
# インデックス位置0にある重みパラメータ
print(model.layers[0].weight)

In [ ]:
# 形状
print(model.layers[0].weight.shape)

In [ ]:
# フィードフォワードパスの挙動
torch.manual_seed(123)
x = torch.rand((1, 50))
out = model(x)
print(out)

In [ ]:
# torch.no_grad()で勾配の追跡を停止
# メモリリソースや計算リソースが大幅に削減される
with torch.no_grad():
    out = model(x)
print(out)

In [ ]:
# 予測値のクラス所属確率の計算
with torch.no_grad():
    out = torch.softmax(model(x), dim=1)
print(out)

## A6. 効率的なデータローダーをセットアップする

In [ ]:
# 単純なデータセットを作成する
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [ ]:
# カスタムDatasetクラスを定義する
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        # データレコードと対応するラベルを1つだけ取得
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        # データセットの全体的な長さ
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
print(len(train_ds))